In [ ]:
import os
import json
import time
import httpx
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("GLM_API_KEY")
BASE_URL = "https://agent-api.bigmodel.cn/api/agent/managed"
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
    "zai-version": "2026-05-26",
    "zai-beta": "managed-agents-2026-09-22"
}

# 1. Загружаем ID агентов из файла
with open("agent_ids.json", "r", encoding="utf-8") as f:
    ids = json.load(f)
COORDINATOR_ID = ids["coordinator_id"]
print(f"Используем координатора: {COORDINATOR_ID}")

# 2. Создаём новую сессию
print("Создаем сессию...")
session_resp = httpx.post(
    f"{BASE_URL}/v1/sessions",
    headers=HEADERS,
    json={
        "agent": COORDINATOR_ID,
        "title": "Power BI Task"
    }
)
session_resp.raise_for_status()
session_id = session_resp.json()["id"]
print(f"Сессия создана: {session_id}")

# 3. Формулируем задачу
task = """
Создай новый дашборд продаж на основе таблиц orders и products из PostgreSQL.
Используй файл C:\\Users\\maxim\\Desktop\\PowerBiAgent\\data_dictionary.md для понимания схемы.
После создания проверь меру 'Total Revenue' и при необходимости попроси model-optimizer оптимизировать её.
"""

# 4. Отправляем задачу
print(f"\nОтправляем задачу...\n{task}")
msg_resp = httpx.post(
    f"{BASE_URL}/v1/sessions/{session_id}/events",
    headers=HEADERS,
    json={"type": "user_message", "content": task}
)
msg_resp.raise_for_status()

# 5. Ожидаем завершения (упрощённый опрос)
print("\nКоординатор работает. Ожидание...")
for i in range(60):
    time.sleep(10)
    events_resp = httpx.get(f"{BASE_URL}/v1/sessions/{session_id}/events", headers=HEADERS)
    events = events_resp.json().get("data", [])
    for event in reversed(events):
        if event.get("type") == "session.status_idle":
            print("\n✅ Задача завершена (сессия перешла в idle).")
            exit()
        if event.get("type") == "run_failed":
            print("\n❌ Ошибка выполнения.")
            exit()
    print(f"   ... проверка {i+1}")